In [1]:
import os
import sys
import yaml
import pandas as pd
import numpy as np
from string import Template
from IPython.display import Markdown, display
import json
import scipy

with open('../../config.local.yaml', 'r') as f:
    local_config = yaml.safe_load(f)

LOCAL_PATH = local_config['LOCAL_PATH']

sys.path.append(os.path.join(LOCAL_PATH, "src/python"))

import writing_tools as wt
import llm_tools as llt


from matplotlib import pyplot as plt
from utils import parse_casenum, weighted_mean, extract_json

plt.rcParams['font.family'] = 'Times New Roman'
plt.rcParams['font.size'] = 11

os.environ['LOKY_MAX_CPU_COUNT'] = '1' # because of windows core count warning

with open('../../config.local.yaml', 'r') as f:
    local_config = yaml.safe_load(f)
with open('../../config.yaml', 'r') as f:
    config = yaml.safe_load(f)

LOCAL_PATH = local_config['LOCAL_PATH']
DATA_PATH = local_config['DATA_PATH']
EMBEDDING_DIMENSION = config['EMBEDDING_DIMENSION']

rng = np.random.default_rng(12898)


In [2]:
MODEL = "claude-opus-4-6"
LLM_PROVIDER = "claude" # openai | claude
MAX_OUTPUT_TOKENS = 8000
input_cost = 2.50 # batch cost per 1M tokens
output_cost = 12.50 # batch cost per 1M tokens
output_tokens_per_request = 4000
REQUESTS_PER_BATCH = 1000
REPLACE = False

In [3]:
rel_path = os.path.join(DATA_PATH, "response_store")
batch_db_path = os.path.join(rel_path, "batch_jobs.db")
response_db_path = os.path.join(rel_path, "responses.db")

batch_db_conn = llt.get_batch_jobs_db_conn(batch_db_path)
response_db_conn = llt.get_response_store_db_conn(response_db_path)


In [4]:
df = pd.read_parquet(os.path.join(DATA_PATH, "intermediate_data/cpc", "analysis_data_w_embeddings.parquet"))

In [5]:
# Flat df of cases with lists of hearing dates and hearing results

flat_df = {}
df['canonical_casenum'] = df['title'].apply(lambda x: parse_casenum(x)['canonical_casenum'])
for idx, row in df.iterrows():
    date = pd.to_datetime(row['date'])
    project_result = row["project_result"]
    canonical_casenum = row['canonical_casenum']
    casenum = row['title']
    if canonical_casenum not in flat_df:
        flat_df[canonical_casenum] = {
            "hearing_dates": [],
            "project_results": [],
            "casenums": [],
            "agenda_texts": [],
            "minutes_summaries": [],
            "consent_calendars": []
        }
    flat_df[canonical_casenum]["hearing_dates"].append(date)
    flat_df[canonical_casenum]["project_results"].append(project_result)
    flat_df[canonical_casenum]["casenums"].append(casenum)
    flat_df[canonical_casenum]["agenda_texts"].append(row["agenda_text"])
    flat_df[canonical_casenum]["minutes_summaries"].append(row["minutes_summary"])
    flat_df[canonical_casenum]["consent_calendars"].append(row["is_consent_calendar"])

flat_df = pd.DataFrame(flat_df, index=['hearing_dates', 'project_results', 'casenums', 'agenda_texts', 'minutes_summaries', 'consent_calendars']).T.reset_index().rename(columns={'index': 'canonical_casenum'})

In [6]:
# Calculate average interval between hearings for cases with multiple hearings

flat_df['n_hearings'] = flat_df['hearing_dates'].apply(len)
flat_df['last_result'] = flat_df['project_results'].apply(lambda x: x[-1])
flat_df['first_result'] = flat_df['project_results'].apply(lambda x: x[0])
flat_df["average_hearing_interval"] = flat_df['hearing_dates'].apply(lambda x: np.mean(np.diff(sorted(x)).astype('timedelta64[D]')) if len(x) > 1 else np.nan)
mask = (flat_df["n_hearings"]>1) & (flat_df["first_result"]=="DELAYED") 
flat_df.loc[mask, "average_hearing_interval"].mean()

Timedelta('56 days 02:10:54.545454545')

In [7]:
# Summarize changes for delayed cases that ultimately get approved

mask = (flat_df["first_result"]=="DELAYED") & (flat_df["n_hearings"]>1)

PROMPT = Template("""
In what follows, I will provide you initial and final hearing information for a single Los Angeles City Planning Commission case. For the initial hearing, the initial agenda and a summary of the outcome are provided. For the final hearing, the final agenda and a summary of the outcome are provided. The outcomes are provided for context only, and I want you to compare the differences between the initial and final agenda items. Return a JSON object with the following structure:

{
    "summary_of_changes": <A summary of the changes between the initial and final hearings, highlighting what was added, removed, or modified in proposed project. Compare the initial and final agenda items only, not the outcomes. If none, indicate that there were no changes.>,
    "degree_of_changes": <A description of the degree of change between the initial project proposal and the one submitted at final hearing. Compare the initial and final agenda items only, not the outcomes. Your options are: "NONE", "MINOR", "MAJOR". If the changes are in the wording only, say "NONE".>,
    "types_of_changes": [<A list of the types of changes between the initial project proposal and the one submitted at final hearing. Compare the initial and final agenda items only, not the outcomes. Your options are: "EXPAND SCOPE OR SCALE", "REDUCE SCOPE OR SCALE", "ADD REQUESTS", "REMOVE REQUESTS", "ADD OR REMOVE DETAILS", "PROJECT APPEALED">]. If none are applicable, return an empty list.
}

<initial_agenda>
$initial_agenda
</initial_agenda>

<initial_outcome>
$initial_outcome
</initial_outcome>

<final_agenda>
$final_agenda
</final_agenda>

<final_outcome>
$final_outcome
</final_outcome>
""")

for idx, row in flat_df.loc[mask].iterrows():
    initial_agenda = row["agenda_texts"][0]
    final_agenda = row["agenda_texts"][-1]
    initial_outcome = row["minutes_summaries"][0]
    final_outcome = row["minutes_summaries"][-1]
    prompt = PROMPT.substitute(
        initial_agenda=initial_agenda,
        final_agenda=final_agenda,
        initial_outcome=initial_outcome,
        final_outcome=final_outcome
    )
    response = llt.get_response(prompt, MODEL, response_db_conn, overwrite=REPLACE, provider=LLM_PROVIDER, max_output_tokens=MAX_OUTPUT_TOKENS)
    j = extract_json(response["response"])
    flat_df.at[idx, "summary_of_changes"] = j["summary_of_changes"]
    flat_df.at[idx, "degree_of_changes"] = j["degree_of_changes"]
    flat_df.at[idx, "types_of_changes"] = j["types_of_changes"]

flat_df["types_of_changes"] = flat_df["types_of_changes"].apply(lambda x: x if isinstance(x, list) else [])

flat_df.to_parquet(os.path.join(DATA_PATH, "intermediate_data/cpc", "delay_impacts_flat_df.parquet"))


In [8]:
# Calculate statistics for never-delayed, non-consent-calendar cases

mask = (flat_df["first_result"]!="DELAYED") & (flat_df["consent_calendars"].apply(lambda x: x[0])==False)

NumNeverDelayedNonCC = mask.sum()

NeverDelayedApproved = (flat_df.loc[mask, "last_result"]=="APPROVED").sum()
NeverDelayedApprovedPct = NeverDelayedApproved / NumNeverDelayedNonCC

NeverDelayedApprovedMinor = (flat_df.loc[mask, "last_result"]=="APPROVED WITH MINOR CONDITIONS OR MODIFICATIONS").sum()
NeverDelayedApprovedMinorPct = NeverDelayedApprovedMinor / NumNeverDelayedNonCC

NeverDelayedApprovedMajor = (flat_df.loc[mask, "last_result"]=="APPROVED WITH MAJOR CONDITIONS OR MODIFICATIONS").sum()
NeverDelayedApprovedMajorPct = NeverDelayedApprovedMajor / NumNeverDelayedNonCC

NeverDelayedDenied = (flat_df.loc[mask, "last_result"].isin(["DENIED", "APPLICATION WITHDRAWN"])).sum()
NeverDelayedDeniedPct = NeverDelayedDenied / NumNeverDelayedNonCC

RESULTS = {
    "NumNeverDelayedNonCC": f"{NumNeverDelayedNonCC:,.0f}",
    "NeverDelayedApproved": f"{NeverDelayedApproved:,.0f}",
    "NeverDelayedApprovedPct": f"{NeverDelayedApprovedPct*100:.1f}",
    "NeverDelayedApprovedMinor": f"{NeverDelayedApprovedMinor:,.0f}",
    "NeverDelayedApprovedMinorPct": f"{NeverDelayedApprovedMinorPct*100:.1f}",
    "NeverDelayedApprovedMajor": f"{NeverDelayedApprovedMajor:,.0f}",
    "NeverDelayedApprovedMajorPct": f"{NeverDelayedApprovedMajorPct*100:.1f}",
    "NeverDelayedDenied": f"{NeverDelayedDenied:,.0f}",
    "NeverDelayedDeniedPct": f"{NeverDelayedDeniedPct*100:.1f}"
}

RESULTS

{'NumNeverDelayedNonCC': '431',
 'NeverDelayedApproved': '204',
 'NeverDelayedApprovedPct': '47.3',
 'NeverDelayedApprovedMinor': '206',
 'NeverDelayedApprovedMinorPct': '47.8',
 'NeverDelayedApprovedMajor': '15',
 'NeverDelayedApprovedMajorPct': '3.5',
 'NeverDelayedDenied': '6',
 'NeverDelayedDeniedPct': '1.4'}

In [9]:
# Calculate statistics for delayed cases

NumDelayedCases = (flat_df["first_result"]=="DELAYED").sum()
NumResolved = ((flat_df["first_result"]=="DELAYED") & (flat_df["last_result"]!="DELAYED")).sum()
NumResolvedPct = NumResolved / NumDelayedCases

mask = (flat_df["first_result"]=="DELAYED") & (flat_df["last_result"]!="DELAYED")

DaysToResolve = flat_df.loc[mask, "hearing_dates"].apply(lambda x: (max(x) - min(x)).days).mean()

ResolvedApproved = (flat_df.loc[mask, "last_result"]=="APPROVED").sum()
ResolvedApprovedPct = ResolvedApproved / NumResolved

ResolvedApprovedMinor = (flat_df.loc[mask, "last_result"]=="APPROVED WITH MINOR CONDITIONS OR MODIFICATIONS").sum()
ResolvedApprovedMinorPct = ResolvedApprovedMinor / NumResolved

ResolvedApprovedMajor = (flat_df.loc[mask, "last_result"]=="APPROVED WITH MAJOR CONDITIONS OR MODIFICATIONS").sum()
ResolvedApprovedMajorPct = ResolvedApprovedMajor / NumResolved

ResolvedDeniedWithdrawn = (flat_df.loc[mask, "last_result"].isin(["DENIED", "APPLICATION WITHDRAWN"])).sum()
ResolvedDeniedWithdrawnPct = ResolvedDeniedWithdrawn / NumResolved

ResolvedNoChanges = (flat_df.loc[mask, "degree_of_changes"]=="NONE").sum()
ResolvedNoChangesPct = ResolvedNoChanges / NumResolved

ResolvedMinorChanges = (flat_df.loc[mask, "degree_of_changes"]=="MINOR").sum()
ResolvedMinorChangesPct = ResolvedMinorChanges / NumResolved

ResolvedMajorChanges = (flat_df.loc[mask, "degree_of_changes"]=="MAJOR").sum()
ResolvedMajorChangesPct = ResolvedMajorChanges / NumResolved

flat_df["major_expand"] = (flat_df["types_of_changes"].apply(lambda x: ("EXPAND SCOPE OR SCALE" in x) or ("ADD REQUESTS" in x))) & (flat_df["degree_of_changes"]=="MAJOR")
flat_df["major_reduce"] = (flat_df["types_of_changes"].apply(lambda x: ("REDUCE SCOPE OR SCALE" in x) or ("REMOVE REQUESTS" in x))) & (flat_df["degree_of_changes"]=="MAJOR")

ResolvedMajorExpand = flat_df.loc[mask, "major_expand"].sum()
ResolvedMajorExpandPct = ResolvedMajorExpand / NumResolved

ResolvedMajorReduce = flat_df.loc[mask, "major_reduce"].sum()
ResolvedMajorReducePct = ResolvedMajorReduce / NumResolved

RESULTS = {
    "NumDelayedCases": f"{NumDelayedCases:.0f}",
    "NumResolved": f"{NumResolved:.0f}",
    "NumResolvedPct": f"{NumResolvedPct*100:.1f}",
    "DaysToResolve": f"{DaysToResolve:.0f}",
    "ResolvedApproved": f"{ResolvedApproved:.0f}",
    "ResolvedApprovedPct": f"{ResolvedApprovedPct*100:.1f}",
    "ResolvedApprovedMinor": f"{ResolvedApprovedMinor:.0f}",
    "ResolvedApprovedMinorPct": f"{ResolvedApprovedMinorPct*100:.1f}",
    "ResolvedApprovedMajor": f"{ResolvedApprovedMajor:.0f}",
    "ResolvedApprovedMajorPct": f"{ResolvedApprovedMajorPct*100:.1f}",
    "ResolvedDeniedWithdrawn": f"{ResolvedDeniedWithdrawn:.0f}",
    "ResolvedDeniedWithdrawnPct": f"{ResolvedDeniedWithdrawnPct*100:.1f}",
    "ResolvedNoChanges": f"{ResolvedNoChanges:.0f}",
    "ResolvedNoChangesPct": f"{ResolvedNoChangesPct*100:.1f}",
    "ResolvedMinorChanges": f"{ResolvedMinorChanges:.0f}",
    "ResolvedMinorChangesPct": f"{ResolvedMinorChangesPct*100:.1f}",
    "ResolvedMajorChanges": f"{ResolvedMajorChanges:.0f}",
    "ResolvedMajorChangesPct": f"{ResolvedMajorChangesPct*100:.1f}",
    "ResolvedMajorExpand": f"{ResolvedMajorExpand:.0f}",
    "ResolvedMajorExpandPct": f"{ResolvedMajorExpandPct*100:.1f}",
    "ResolvedMajorReduce": f"{ResolvedMajorReduce:.0f}",
    "ResolvedMajorReducePct": f"{ResolvedMajorReducePct*100:.1f}",
}

_ = wt.update_results(RESULTS)

RESULTS




{'NumDelayedCases': '71',
 'NumResolved': '66',
 'NumResolvedPct': '93.0',
 'DaysToResolve': '85',
 'ResolvedApproved': '21',
 'ResolvedApprovedPct': '31.8',
 'ResolvedApprovedMinor': '32',
 'ResolvedApprovedMinorPct': '48.5',
 'ResolvedApprovedMajor': '8',
 'ResolvedApprovedMajorPct': '12.1',
 'ResolvedDeniedWithdrawn': '5',
 'ResolvedDeniedWithdrawnPct': '7.6',
 'ResolvedNoChanges': '47',
 'ResolvedNoChangesPct': '71.2',
 'ResolvedMinorChanges': '13',
 'ResolvedMinorChangesPct': '19.7',
 'ResolvedMajorChanges': '6',
 'ResolvedMajorChangesPct': '9.1',
 'ResolvedMajorExpand': '5',
 'ResolvedMajorExpandPct': '7.6',
 'ResolvedMajorReduce': '2',
 'ResolvedMajorReducePct': '3.0'}

In [10]:
# z-test

n1 = NumNeverDelayedNonCC
p1 = NeverDelayedApprovedPct

n2 = NumResolved
p2 = ResolvedApprovedPct

pooled_p = (ResolvedApproved + NeverDelayedApproved) / (NumResolved + NumNeverDelayedNonCC)
standard_error = np.sqrt(pooled_p * (1 - pooled_p) * (1/n1 + 1/n2))
z_score = (p1 - p2) / standard_error

p_value = 2 * (1 - scipy.stats.norm.cdf(abs(z_score)))

print(p_value)


0.018378528740058186


In [11]:
# Make table

header = r"""
\begin{table}[H]
\centering
\caption{What Happens to Delayed Cases?}
\vspace{0.2cm}
\label{tab_delayed_outcomes}
\begin{threeparttable}
\begin{tabular}{lrr} \toprule
 """
footer = r"""
\bottomrule
\end{tabular}
\begin{tablenotes}
\item {\textit{Notes: } This table shows what happens to cases that get delayed in our data. Note that outcomes are tracked between the first observed hearing and the last observed hearing, though there may be additional hearings in between.}
\end{tablenotes}
\end{threeparttable}
\end{table}
"""
tbl = ""
tbl += fr"Num Delayed Cases & {NumDelayedCases:.0f}  & \\ [1ex]" + "\n"
tbl += fr"Num Resolved      & {NumResolved:.0f}  & ({NumResolvedPct*100:.1f}\%) \\ [1ex]" + "\n"
tbl += fr"~ ~ Avg. Time to Resolution & {DaysToResolve:.0f} days & \\ [1ex]" + "\n"
tbl += fr"~ ~ Result & & \\ [1ex]" + "\n"
tbl += fr"~ ~ ~ ~ Approved & {ResolvedApproved:.0f}  & ({ResolvedApprovedPct*100:.1f}\%) \\ [1ex]" + "\n"
tbl += fr"~ ~ ~ ~ Approved with Minor Conditions & {ResolvedApprovedMinor:.0f}  & ({ResolvedApprovedMinorPct*100:.1f}\%) \\ [1ex]" + "\n"
tbl += fr"~ ~ ~ ~ Approved with Major Conditions & {ResolvedApprovedMajor:.0f}  & ({ResolvedApprovedMajorPct*100:.1f}\%) \\ [1ex]" + "\n"
tbl += fr"~ ~ ~ ~ Denied or Withdrawn & {ResolvedDeniedWithdrawn:.0f}  & ({ResolvedDeniedWithdrawnPct*100:.1f}\%) \\ [1ex]" + "\n"
tbl += fr"~ ~ Changes to Application & & \\ [1ex]" + "\n"
tbl += fr"~ ~ ~ ~ No Changes & {ResolvedNoChanges:.0f}  & ({ResolvedNoChangesPct*100:.1f}\%) \\ [1ex]" + "\n"
tbl += fr"~ ~ ~ ~ Minor Changes & {ResolvedMinorChanges:.0f}  & ({ResolvedMinorChangesPct*100:.1f}\%) \\ [1ex]" + "\n"
tbl += fr"~ ~ ~ ~ Major Changes & {ResolvedMajorChanges:.0f}  & ({ResolvedMajorChangesPct*100:.1f}\%) \\ [1ex]" + "\n"

print(header + tbl + footer)

with open(os.path.join(LOCAL_PATH, "tables", "tab_delayed_outcomes.tex"), "w", encoding='utf-8') as f:
    f.write(header + tbl + footer)





\begin{table}[H]
\centering
\caption{What Happens to Delayed Cases?}
\vspace{0.2cm}
\label{tab_delayed_outcomes}
\begin{threeparttable}
\begin{tabular}{lrr} \toprule
 Num Delayed Cases & 71  & \\ [1ex]
Num Resolved      & 66  & (93.0\%) \\ [1ex]
~ ~ Avg. Time to Resolution & 85 days & \\ [1ex]
~ ~ Result & & \\ [1ex]
~ ~ ~ ~ Approved & 21  & (31.8\%) \\ [1ex]
~ ~ ~ ~ Approved with Minor Conditions & 32  & (48.5\%) \\ [1ex]
~ ~ ~ ~ Approved with Major Conditions & 8  & (12.1\%) \\ [1ex]
~ ~ ~ ~ Denied or Withdrawn & 5  & (7.6\%) \\ [1ex]
~ ~ Changes to Application & & \\ [1ex]
~ ~ ~ ~ No Changes & 47  & (71.2\%) \\ [1ex]
~ ~ ~ ~ Minor Changes & 13  & (19.7\%) \\ [1ex]
~ ~ ~ ~ Major Changes & 6  & (9.1\%) \\ [1ex]

\bottomrule
\end{tabular}
\begin{tablenotes}
\item {\textit{Notes: } This table shows what happens to cases that get delayed in our data. Note that outcomes are tracked between the first observed hearing and the last observed hearing, though there may be additional hearings i